In [0]:
%scala
// in Scala
val person = Seq(
    (0, "Bill Chambers", 0, Seq(100)),
    (1, "Matei Zaharia", 1, Seq(500, 250, 100)),
    (2, "Michael Armbrust", 1, Seq(250, 100)))
  .toDF("id", "name", "graduate_program", "spark_status")
val graduateProgram = Seq(
    (0, "Masters", "School of Information", "UC Berkeley"),
    (2, "Masters", "EECS", "UC Berkeley"),
    (1, "Ph.D.", "EECS", "UC Berkeley"))
  .toDF("id", "degree", "department", "school")
val sparkStatus = Seq(
    (500, "Vice President"),
    (250, "PMC Member"),
    (100, "Contributor"))
  .toDF("id", "status")


person: org.apache.spark.sql.DataFrame = [id: int, name: string ... 2 more fields]
graduateProgram: org.apache.spark.sql.DataFrame = [id: int, degree: string ... 2 more fields]
sparkStatus: org.apache.spark.sql.DataFrame = [id: int, status: string]

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * 
spark = SparkSession.builder.appName("Joins").getOrCreate()

In [0]:
data = [
    (0, "Bill Chambers", 0, [100]),
    (1, "Matei Zaharia", 1, [500, 250, 100]),
    (2, "Michael Armbrust", 1, [250, 100] ) ]
col = ["id", "name", "graduate_program", "spark_status"]
df_p = spark.createDataFrame(data, colP)
data = [
    (0, "Masters", "School of Information", "UC Berkeley"),
    (2, "Masters", "EECS", "UC Berkeley"),
    (1, "Ph.D.", "EECS", "UC Berkeley")]
col = ["id", "degree", "department", "school"]
df_gp = spark.createDataFrame(data, col)
data = [
    (500, "Vice President"),
    (250, "PMC Member"),
    (100, "Contributor") ]
col = ("id", "status")
df_ss = spark.createDataFrame(data,col)



In [0]:
%scala
person.createOrReplaceTempView("person")
graduateProgram.createOrReplaceTempView("graduateProgram")
sparkStatus.createOrReplaceTempView("sparkStatus")


In [0]:
%scala
val person = spark.sql("SELECT * FROM person")
display(person)

id,name,graduate_program,spark_status
0,Bill Chambers,0,List(100)
1,Matei Zaharia,1,"List(500, 250, 100)"
2,Michael Armbrust,1,"List(250, 100)"


In [0]:
df_p.show()

+---+----------------+----------------+---------------+
| id|            name|graduate_program|   spark_status|
+---+----------------+----------------+---------------+
|  0|   Bill Chambers|               0|          [100]|
|  1|   Matei Zaharia|               1|[500, 250, 100]|
|  2|Michael Armbrust|               1|     [250, 100]|
+---+----------------+----------------+---------------+



In [0]:
%scala
display(spark.sql("SELECT * FROM graduateProgram"))

In [0]:
df_gp.show()

+---+-------+--------------------+-----------+
| id| degree|          department|     school|
+---+-------+--------------------+-----------+
|  0|Masters|School of Informa...|UC Berkeley|
|  2|Masters|                EECS|UC Berkeley|
|  1|  Ph.D.|                EECS|UC Berkeley|
+---+-------+--------------------+-----------+



In [0]:
%scala
display(spark.sql("SELECT * FROM sparkStatus"))

In [0]:
df_ss.show()

+---+--------------+
| id|        status|
+---+--------------+
|500|Vice President|
|250|    PMC Member|
|100|   Contributor|
+---+--------------+



In [0]:
%scala
// in Scala
val joinExpression = person.col("graduate_program") === graduateProgram.col("id")


joinExpression: org.apache.spark.sql.Column = (graduate_program = id)

In [0]:
df_join_inner = df_p.join(df_gp, df_p.graduate_program==df_gp.id,"inner")
df_join_inner.show()
df_join_inner.explain()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [graduate_program#20L], [id#26L], Inner
   :- Sort [graduate_program#20L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(graduate_program#20L, 200), ENSURE_REQUIREMENTS, [plan_id=399]
   :     +- Filter is

Adaptive Query Execution – AQE

- Spark czyta dane z df_p (tabela person)
- Filtruje NULLe w kolumnie graduate_program
- Haszuje dane na 200 partycji (do JOINa)
- Sortuje po graduate_program, żeby przygotować dane do SortMergeJoin


In [0]:
df_join_left = df_p.join(df_gp, df_p.graduate_program==df_gp.id,"left")
df_join_left.show()
df_join_left.explain()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [graduate_program#20L], [id#26L], LeftOuter
   :- Sort [graduate_program#20L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(graduate_program#20L, 200), ENSURE_REQUIREMENTS, [plan_id=632]
   :     +- Scan 

- Partycjonowanie (hashpartitioning) po kluczach
- Sortowanie (Sort)
- JOIN: tym razem jako LeftOuter
- Filtrowanie nulli tylko po stronie df_gp (prawa), bo lewa strona ma zostać cała

In [0]:
df_join_left = df_p.join(df_gp, df_p.graduate_program==df_gp.id,"right")
df_join_left.show()
df_join_left.explain()

+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+
|   0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|null|            null|            null|           null|  2|Masters|                EECS|UC Berkeley|
|   2|Michael Armbrust|               1|     [250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|   1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
+----+----------------+----------------+---------------+---+-------+--------------------+-----------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [graduate_program#20L], [id#26L], RightOuter
   :- Sort [graduate_program#20L ASC NULLS FIRST], false, 0
   : 

- Partycjonowanie (hashpartitioning) po kluczach
- Sortowanie (Sort)
- JOIN: tym razem jako RightOuter
- Filtrowanie nulli tylko po stronie df_gp (lewa), bo prawa strona ma zostać cała

In [0]:
df_join_left = df_p.join(df_gp, df_p.graduate_program==df_gp.id,"left_semi")
df_join_left.show()
df_join_left.explain()

+---+----------------+----------------+---------------+
| id|            name|graduate_program|   spark_status|
+---+----------------+----------------+---------------+
|  0|   Bill Chambers|               0|          [100]|
|  1|   Matei Zaharia|               1|[500, 250, 100]|
|  2|Michael Armbrust|               1|     [250, 100]|
+---+----------------+----------------+---------------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [graduate_program#20L], [id#26L], LeftSemi
   :- Sort [graduate_program#20L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(graduate_program#20L, 200), ENSURE_REQUIREMENTS, [plan_id=1084]
   :     +- Filter isnotnull(graduate_program#20L)
   :        +- Scan ExistingRDD[id#18L,name#19,graduate_program#20L,spark_status#21]
   +- Sort [id#26L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(id#26L, 200), ENSURE_REQUIREMENTS, [plan_id=1085]
         +- Project [id#26L]
            +- Filter isnotnull(id#2

- Spark wykona Sort Merge Join w trybie LeftSemi, czyli:
- Zwróci tylko wiersze z lewej tabeli (df_p), które mają dopasowanie w prawej (df_gp)
- Nie dołączy żadnych kolumn z df_gp

In [0]:
df_join_left = df_gp.join(df_p, df_gp.id==df_p.graduate_program,"left_anti")
df_join_left.show()
df_join_left.explain()

+---+-------+----------+-----------+
| id| degree|department|     school|
+---+-------+----------+-----------+
|  2|Masters|      EECS|UC Berkeley|
+---+-------+----------+-----------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [id#26L], [graduate_program#20L], LeftAnti
   :- Sort [id#26L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(id#26L, 200), ENSURE_REQUIREMENTS, [plan_id=1543]
   :     +- Scan ExistingRDD[id#26L,degree#27,department#28,school#29]
   +- Sort [graduate_program#20L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(graduate_program#20L, 200), ENSURE_REQUIREMENTS, [plan_id=1544]
         +- Project [graduate_program#20L]
            +- Filter isnotnull(graduate_program#20L)
               +- Scan ExistingRDD[id#18L,name#19,graduate_program#20L,spark_status#21]




- Zwraca tylko te wiersze z df_p, które nie mają dopasowania w df_gp po graduate_program == id
- Żadnych kolumn z df_gp
- Typowe zastosowanie: "pokaż mi to, czego brakuje", np. osoby, które nie są przypisane do żadnego programu



In [0]:
df_cross = df_p.crossJoin(df_gp)
df_cross.show()
df_cross.explain()

+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
| id|            name|graduate_program|   spark_status| id| degree|          department|     school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
|  0|   Bill Chambers|               0|          [100]|  0|Masters|School of Informa...|UC Berkeley|
|  0|   Bill Chambers|               0|          [100]|  2|Masters|                EECS|UC Berkeley|
|  0|   Bill Chambers|               0|          [100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  0|Masters|School of Informa...|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  2|Masters|                EECS|UC Berkeley|
|  1|   Matei Zaharia|               1|[500, 250, 100]|  1|  Ph.D.|                EECS|UC Berkeley|
|  2|Michael Armbrust|               1|     [250, 100]|  0|Masters|School of Informa...|UC 

- Iloczyn kartezjański 
- kazdy rekord z lewej do każdego rekordu z prawej

In [0]:
%scala
// in Scala
val wrongJoinExpression = person.col("name") === graduateProgram.col("school")


wrongJoinExpression: org.apache.spark.sql.Column = (name = school)

In [0]:
%scala
person.join(graduateProgram, joinExpression).show()


+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
 id| name|graduate_program| spark_status| id| degree| department| school|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+
 0| Bill Chambers| 0| [100]| 0|Masters|School of Informa...|UC Berkeley|
 2|Michael Armbrust| 1| [250, 100]| 1| Ph.D.| EECS|UC Berkeley|
 1| Matei Zaharia| 1|[500, 250, 100]| 1| Ph.D.| EECS|UC Berkeley|
+---+----------------+----------------+---------------+---+-------+--------------------+-----------+

In [0]:
%scala
person.join(graduateProgram, wrongJoinExpression).show()

In [0]:
%scala
// in Scala
var joinType = "inner"
person.join(graduateProgram, joinExpression, joinType).explain

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [graduate_program#564], [id#583], Inner, BuildLeft, false, true
 :- Exchange SinglePartition, EXECUTOR_BROADCAST, [plan_id=1676]
 : +- LocalTableScan [id#562, name#563, graduate_program#564, spark_status#565]
 +- LocalTableScan [id#583, degree#584, department#585, school#586]


joinType: String = inner

In [0]:
%scala
joinType = "outer"
person.join(graduateProgram, joinExpression, joinType).show()


In [0]:
%scala
joinType = "left_outer"
graduateProgram.join(person, joinExpression, joinType).show()


In [0]:
%scala
joinType = "right"
person.join(graduateProgram, joinExpression, joinType).show()


In [0]:
%scala
joinType = "left_semi"
graduateProgram.join(person, joinExpression, joinType).show()


In [0]:
%scala
// in Scala
val gradProgram2 = graduateProgram.union(Seq(
    (0, "Masters", "Duplicated Row", "Duplicated School")).toDF())

gradProgram2.createOrReplaceTempView("gradProgram2")


gradProgram2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [id: int, degree: string ... 2 more fields]

In [0]:
%scala
gradProgram2.join(person, joinExpression, joinType).show()


+---+-------+--------------------+-----------------+---+----------------+----------------+---------------+
 id| degree| department| school| id| name|graduate_program| spark_status|
+---+-------+--------------------+-----------------+---+----------------+----------------+---------------+
 0|Masters|School of Informa...| UC Berkeley| 0| Bill Chambers| 0| [100]|
 1| Ph.D.| EECS| UC Berkeley| 2|Michael Armbrust| 1| [250, 100]|
 1| Ph.D.| EECS| UC Berkeley| 1| Matei Zaharia| 1|[500, 250, 100]|
 0|Masters| Duplicated Row|Duplicated School| 0| Bill Chambers| 0| [100]|
+---+-------+--------------------+-----------------+---+----------------+----------------+---------------+

In [0]:
%scala
joinType = "left_anti"
graduateProgram.join(person, joinExpression, joinType).show()


In [0]:
%scala
joinType = "cross"
graduateProgram.join(person, joinExpression, joinType).show()


In [0]:
%scala

graduateProgram.crossJoin(person).show()

In [0]:
%scala
graduateProgram.join(person, joinExpression, joinType).explain

In [0]:
%scala
graduateProgram.crossJoin(person).explain

In [0]:
%scala
import org.apache.spark.sql.functions.expr

person.withColumnRenamed("id", "personId")
  .join(sparkStatus, expr("array_contains(spark_status, id)")).show()


### Duplikaty

In [0]:
%scala
val gradProgramDupe = graduateProgram.withColumnRenamed("id", "graduate_program")


gradProgramDupe: org.apache.spark.sql.DataFrame = [graduate_program: int, degree: string ... 2 more fields]

In [0]:
%scala
display(gradProgramDupe)

graduate_program,degree,department,school
0,Masters,School of Information,UC Berkeley
2,Masters,EECS,UC Berkeley
1,Ph.D.,EECS,UC Berkeley


In [0]:
%scala
val joinExpr = gradProgramDupe.col("graduate_program") === person.col("graduate_program")


joinExpr: org.apache.spark.sql.Column = (graduate_program = graduate_program)

In [0]:
%scala
person.join(gradProgramDupe, joinExpr).show()


+---+----------------+----------------+---------------+----------------+-------+--------------------+-----------+
 id| name|graduate_program| spark_status|graduate_program| degree| department| school|
+---+----------------+----------------+---------------+----------------+-------+--------------------+-----------+
 0| Bill Chambers| 0| [100]| 0|Masters|School of Informa...|UC Berkeley|
 2|Michael Armbrust| 1| [250, 100]| 1| Ph.D.| EECS|UC Berkeley|
 1| Matei Zaharia| 1|[500, 250, 100]| 1| Ph.D.| EECS|UC Berkeley|
+---+----------------+----------------+---------------+----------------+-------+--------------------+-----------+

In [0]:

person.join(gradProgramDupe, joinExpr).select("graduate_program").show()


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-1242020124628898>:1
----> 1 person.join(gradProgramDupe, joinExpr).select("graduate_program").show()

NameError: name 'person' is not defined

###Opcja 1

In [0]:
%scala
person.join(gradProgramDupe,"graduate_program").select("graduate_program").show()


### Opcja 2 Drop after join

In [0]:
%scala
person.join(gradProgramDupe, joinExpr).drop(person.col("graduate_program"))
  .select("graduate_program").show()


In [0]:
%scala
val joinExpr = person.col("graduate_program") === graduateProgram.col("id")
person.join(graduateProgram, joinExpr).drop(graduateProgram.col("id")).show()


### Opcja 3

In [0]:
%scala
val gradProgram3 = graduateProgram.withColumnRenamed("id", "grad_id")
val joinExpr = person.col("graduate_program") === gradProgram3.col("grad_id")
person.join(gradProgram3, joinExpr).show()
